 Data Preprocessing | Forest Fires Capstone

**TKH Phase 2 Capstone Project**

This notebook prepares the Forest Fires dataset for the unsupervised and supervised modeling stages.

### Preprocessing goals
- Load the raw dataset used in EDA.
- Remove exact duplicate rows.
- Confirm missing values and data types.
- Create the regression target `log_area` using `np.log1p(area)`.
- Encode the categorical `month` and `day` variables.
- Define the predictor columns and target clearly.
- Save a clean processed dataset to `data/processed/forestfires_processed.csv`.



## 1. Imports

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 2. Load the Raw Dataset

The path logic below works with the recommended GitHub structure and also with a local copy of the CSV.

In [3]:
# Recommended project path first; local fallbacks make the notebook portable.
possible_paths = [
    Path("../data/raw/forestfires.csv"),
    Path("data/raw/forestfires.csv"),
    Path("forestfires.csv"),
    Path("/mnt/data/forestfires.csv"),
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "forestfires.csv was not found. Place it in data/raw/ or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Original shape: {df.shape}")
df.head()

Loaded: ../data/raw/forestfires.csv
Original shape: (517, 13)


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.200,26.200,94.300,5.100,8.200,51,6.700,0.000,0.000
1,7,4,oct,tue,90.600,35.400,669.100,6.700,18.000,33,0.900,0.000,0.000
2,7,4,oct,sat,90.600,43.700,686.900,6.700,14.600,33,1.300,0.000,0.000
3,8,6,mar,fri,91.700,33.300,77.500,9.000,8.300,97,4.000,0.200,0.000
4,8,6,mar,sun,89.300,51.300,102.200,9.600,11.400,99,1.800,0.000,0.000


## 3. Validate the Dataset Before Cleaning

Before changing the data, verify that the expected columns are present and inspect missing values and duplicates.

In [4]:
expected_columns = [
    "X", "Y", "month", "day", "FFMC", "DMC", "DC", "ISI",
    "temp", "RH", "wind", "rain", "area"
]

missing_columns = [col for col in expected_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))

print(f"Exact duplicate rows: {df.duplicated().sum()}")

Data types:


,dtype
X,int64
Y,int64
month,str
day,str
FFMC,float64
DMC,float64
DC,float64
ISI,float64
temp,float64
RH,int64


Missing values:


,missing_count
X,0
Y,0
month,0
day,0
FFMC,0
DMC,0
DC,0
ISI,0
temp,0
RH,0


Exact duplicate rows: 4


## 4. Remove Exact Duplicate Rows

EDA identified **4 exact duplicate rows**. Exact duplicates do not add new information and could slightly overweight repeated observations, so they are removed before modeling.

In [5]:
rows_before = len(df)

df_clean = df.drop_duplicates().reset_index(drop=True).copy()

rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print(f"Rows before:  {rows_before}")
print(f"Rows removed: {rows_removed}")
print(f"Rows after:   {rows_after}")

Rows before:  517
Rows removed: 4
Rows after:   513


## 5. Confirm Missing Values After Cleaning

The original dataset contains no missing values, so no imputation is required.

In [6]:
missing_after = df_clean.isna().sum()
print(f"Total missing values: {missing_after.sum()}")
display(missing_after.to_frame("missing_count"))

Total missing values: 0


,missing_count
X,0
Y,0
month,0
day,0
FFMC,0
DMC,0
DC,0
ISI,0
temp,0
RH,0


## 6. Create the Regression Target: `log_area`

The raw `area` variable is highly right-skewed and contains many zero values. We therefore create:

\[
	ext{log\_area} = \log(1 + 	ext{area})
\]

`np.log1p()` is useful here because it is defined when `area = 0` and compresses very large burned-area values.

The original `area` column is retained for interpretation and Tableau, but **`log_area` will be the primary regression target**.

In [7]:
df_clean["log_area"] = np.log1p(df_clean["area"])

zero_count = (df_clean["area"] == 0).sum()
zero_pct = (df_clean["area"] == 0).mean() * 100

print(f"Zero-area observations after deduplication: {zero_count} ({zero_pct:.1f}%)")
display(df_clean[["area", "log_area"]].describe())

Zero-area observations after deduplication: 244 (47.6%)


,area,log_area
count,513.000,513.000
mean,12.892,1.113
std,63.893,1.398
min,0.000,0.000
25%,0.000,0.000
50%,0.540,0.432
75%,6.570,2.024
max,"1,090.840",6.996


## 7. Encode Categorical Variables

`month` and `day` are categorical text variables. Most machine-learning algorithms require numeric input, so we one-hot encode them.

We keep **all categories** (`drop_first=False`). Tree-based models can use them directly, while linear-model handling will be controlled later through the modeling pipeline.

In [8]:
categorical_cols = ["month", "day"]

encoded_categories = pd.get_dummies(
    df_clean[categorical_cols],
    columns=categorical_cols,
    prefix=categorical_cols,
    dtype=int,
)

print(f"Encoded categorical columns: {encoded_categories.shape[1]}")
print(encoded_categories.columns.tolist())

Encoded categorical columns: 19
['month_apr', 'month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep', 'day_fri', 'day_mon', 'day_sat', 'day_sun', 'day_thu', 'day_tue', 'day_wed']


## 8. Build the Processed Modeling Dataset

The predictor set contains:
- Spatial grid coordinates: `X`, `Y`
- Fire-weather indices: `FFMC`, `DMC`, `DC`, `ISI`
- Weather variables: `temp`, `RH`, `wind`, `rain`
- One-hot encoded month and weekday variables

`area` and `log_area` are **targets/outcomes and are not predictor features**.

In [9]:
numeric_predictors = [
    "X", "Y", "FFMC", "DMC", "DC", "ISI", "temp", "RH", "wind", "rain"
]

processed_df = pd.concat(
    [
        df_clean[numeric_predictors + ["area", "log_area"]],
        encoded_categories,
    ],
    axis=1,
)

feature_columns = numeric_predictors + encoded_categories.columns.tolist()
target_column = "log_area"

X = processed_df[feature_columns].copy()
y = processed_df[target_column].copy()

print(f"Processed dataset shape: {processed_df.shape}")
print(f"Predictor matrix X shape: {X.shape}")
print(f"Target y shape: {y.shape}")
processed_df.head()

Processed dataset shape: (513, 31)
Predictor matrix X shape: (513, 29)
Target y shape: (513,)


,X,Y,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area,log_area,month_apr,month_aug,month_dec,month_feb,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,day_fri,day_mon,day_sat,day_sun,day_thu,day_tue,day_wed
0,7,5,86.200,26.200,94.300,5.100,8.200,51,6.700,0.000,0.000,0.000,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
1,7,4,90.600,35.400,669.100,6.700,18.000,33,0.900,0.000,0.000,0.000,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0
2,7,4,90.600,43.700,686.900,6.700,14.600,33,1.300,0.000,0.000,0.000,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
3,8,6,91.700,33.300,77.500,9.000,8.300,97,4.000,0.200,0.000,0.000,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
4,8,6,89.300,51.300,102.200,9.600,11.400,99,1.800,0.000,0.000,0.000,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0


## 9. Final Quality Checks

Before saving, verify that:
- there are no missing values,
- all modeling predictors are numeric,
- there are no duplicate rows,
- the target is not accidentally included in the feature matrix.

In [10]:
assert processed_df.isna().sum().sum() == 0, "Processed data contains missing values."
assert X.select_dtypes(exclude=np.number).shape[1] == 0, "Non-numeric predictors remain."
assert processed_df.duplicated().sum() == 0, "Duplicate rows remain."
assert "area" not in X.columns and "log_area" not in X.columns, "Target leakage detected."

print("✓ No missing values")
print("✓ All predictors are numeric")
print("✓ No duplicate rows")
print("✓ area/log_area are excluded from X")
print("✓ Preprocessing checks passed")

✓ No missing values
✓ All predictors are numeric
✓ No duplicate rows
✓ area/log_area are excluded from X
✓ Preprocessing checks passed


## 10. Save the Processed Dataset

The notebook saves the processed file to the recommended project location when available. A local fallback is used if the project directory does not exist yet.

In [11]:
# Prefer the GitHub project structure.
project_output_dir = Path("../data/processed")

# If running from the project root instead of notebooks/, use data/processed.
if Path("data").exists() and not Path("../data").exists():
    project_output_dir = Path("data/processed")

# In standalone environments, save beside the notebook/data copy.
if not project_output_dir.parent.exists():
    project_output_dir = Path("/mnt/data") if Path("/mnt/data").exists() else Path(".")

project_output_dir.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = project_output_dir / "forestfires_processed.csv"

processed_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved processed dataset to: {OUTPUT_PATH}")
print(f"Saved shape: {processed_df.shape}")

Saved processed dataset to: ../data/processed/forestfires_processed.csv
Saved shape: (513, 31)


## 11. Preprocessing Summary

After preprocessing:

- Exact duplicate rows were removed.
- No missing-value imputation was necessary.
- `log_area = log(1 + area)` was created as the primary regression target.
- `month` and `day` were one-hot encoded.
- `area` and `log_area` were kept out of the predictor matrix to prevent target leakage.
- The processed dataset was saved for later analysis.

### Decisions carried into the next notebooks

**03 — Unsupervised Learning**  
We will standardize continuous fire/weather features before clustering because variables such as `DC`, `DMC`, `RH`, and `rain` are measured on very different scales. We can test clustering (for example, K-Means) and examine whether the resulting groups represent distinct fire-weather conditions.

**04 — Supervised Modeling**  
We will predict `log_area` and compare 2–3 regression approaches. Train/test splitting and any required scaling will be performed inside the modeling workflow so the test set remains unseen during preprocessing.

> **Important:** A cluster label may later be added as an additional supervised-learning feature, but only after the clustering method has been developed appropriately. This provides a clear connection between the capstone's unsupervised and supervised requirements.

## 12. Variables Available for Modeling

In [12]:
print("Target:")
print(f"  {target_column}")

print(f"\nNumber of predictors: {len(feature_columns)}")
print("Predictors:")
for col in feature_columns:
    print(f"  - {col}")

Target:
  log_area

Number of predictors: 29
Predictors:
  - X
  - Y
  - FFMC
  - DMC
  - DC
  - ISI
  - temp
  - RH
  - wind
  - rain
  - month_apr
  - month_aug
  - month_dec
  - month_feb
  - month_jan
  - month_jul
  - month_jun
  - month_mar
  - month_may
  - month_nov
  - month_oct
  - month_sep
  - day_fri
  - day_mon
  - day_sat
  - day_sun
  - day_thu
  - day_tue
  - day_wed
